# MedScribe-TR — Ses → Metin (Kişi 1)
Doktor–hasta ses kaydı → konuşmacı etiketli transkript (ortak JSON formatı).

**Çalıştırma sırası:** Runtime → T4 GPU seç → 0. hücreyi çalıştır → **Runtime → Restart session** → Run all.
Colab Secrets'ta `HF_TOKEN` tanımlı ve "Notebook access" açık olmalı.

## 0. Kurulum (bir kez; sonra Restart session)

In [11]:
!pip install -q whisperx

## 1. Token ve model erişimi

In [12]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    # Secrets erişimi yoksa (izin kapalı / sekme arka planda) token'ı elle iste
    print("Secrets okunamadı:", type(e).__name__, "-> token'ı elle gir")
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
for repo in ["pyannote/speaker-diarization-community-1"]:
    try:
        api.model_info(repo)
        print("erişim tamam:", repo)
    except Exception as e:
        print("erişim YOK:", repo, "->", type(e).__name__)

erişim tamam: pyannote/speaker-diarization-community-1


## 2. Modül: `asr/transcribe.py`
Kod modül olarak yazılır (GitHub'a bu dosya gider); notebook yalnızca deney kaydıdır.

In [13]:
!mkdir -p /content/medscribe/asr
!touch /content/medscribe/asr/__init__.py

In [14]:
%%writefile /content/medscribe/asr/transcribe.py
"""MedScribe ses -> metin modülü (Kişi 1).

transcribe(audio_path) -> ortak JSON formatında dialogue sözlüğü
(konuşmacı etiketli, zaman damgalı). role / gold_role / role_prob None kalır;
doktor/hasta kararını Kişi 2'nin assign_roles() modülü verir.

Gereksinimler:
  - pip install whisperx
  - ortam değişkeni HF_TOKEN (Hugging Face erişim anahtarı) ve
    pyannote/speaker-diarization-community-1 için kabul edilmiş kullanım şartları
"""
import os

import torch
import whisperx
from whisperx.diarize import DiarizationPipeline

_cache = {}


def _load(language, whisper_model, device):
    """Modelleri bir kez yükler, sonraki çağrılarda önbellekten verir."""
    key = (language, whisper_model, device)
    if key not in _cache:
        compute_type = "float16" if device == "cuda" else "int8"
        asr = whisperx.load_model(whisper_model, device,
                                  compute_type=compute_type, language=language)
        align, meta = whisperx.load_align_model(language_code=language, device=device)
        diar = DiarizationPipeline(token=os.environ["HF_TOKEN"], device=device)
        _cache[key] = (asr, align, meta, diar)
    return _cache[key]


def transcribe(audio_path, language="en", whisper_model="small",
               dialogue_id="asr_001", num_speakers=2, device=None):
    """Ses dosyasını konuşmacı etiketli transkripte çevirir.

    num_speakers: bilinen konuşmacı sayısı (muayenede 2). None verilirse
    diarization modeli sayıyı kendisi seçer.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    asr, align, meta, diar = _load(language, whisper_model, device)
    audio = whisperx.load_audio(audio_path)

    # 1) transkripsiyon, 2) kelime hizalama, 3) konuşmacı ayırma
    result = asr.transcribe(audio, batch_size=16)
    result = whisperx.align(result["segments"], align, meta, audio, device,
                            return_char_alignments=False)
    diarize_segments = diar(audio, min_speakers=num_speakers,
                            max_speakers=num_speakers)
    result = whisperx.assign_word_speakers(diarize_segments, result)

    segments = []
    for s in result["segments"]:
        text = s["text"].strip()
        if not text:  # check_format boş metni reddeder
            continue
        segments.append({
            "id": len(segments),
            "start": round(s["start"], 2),
            "end": round(s["end"], 2),
            "speaker": s.get("speaker", "UNKNOWN"),
            "gold_role": None,
            "role": None,
            "role_prob": None,
            "text": text,
        })
    return {"dialogue_id": dialogue_id, "source": "asr",
            "reference_note": None, "segments": segments}

Overwriting /content/medscribe/asr/transcribe.py


In [15]:
import sys
sys.path.append("/content/medscribe")
from asr.transcribe import transcribe

## 3. Format kontrolü (`check_format`; sonunda `common/format.py`'dan gelecek)

In [16]:
ROLES = {"Doctor", "Patient", None}
TOP = ["dialogue_id", "source", "reference_note", "segments"]
SEG = ["id", "start", "end", "speaker", "gold_role", "role", "role_prob", "text"]

def check_format(d):
    for k in TOP: assert k in d, f"eksik alan: {k}"
    for i, s in enumerate(d["segments"]):
        for k in SEG: assert k in s, f"cümle {i}: eksik alan {k}"
        assert s["id"] == i, f"cümle {i}: id sırası bozuk"
        assert s["gold_role"] in ROLES and s["role"] in ROLES, f"cümle {i}: geçersiz rol"
        assert s["role_prob"] is None or 0 <= s["role_prob"] <= 1, f"cümle {i}: role_prob"
        assert isinstance(s["text"], str) and s["text"].strip(), f"cümle {i}: boş metin"
    return True

## 4. Örnek veri: PriMock57
57 sahte İngilizce muayene; doktor ve hasta ayrı kanallarda. Biz iki kanalı tek dosyada
birleştirip modele tek kanallı kayıt gibi veriyoruz. Doğru cevap (gold) kanal enerjisinden çıkar.

In [17]:
!apt-get -qq install git-lfs > /dev/null
!git lfs install

Git LFS initialized.


In [18]:
import os, subprocess
ROOT = "/content/primock57"
if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/babylonhealth/primock57", ROOT],
                   env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}, check=True)

def prepare_audio(name, seconds=90):
    """İki kanalı çeker, birleştirir; /content/<name>.wav yazar."""
    files = [f"audio/{name}_doctor.wav", f"audio/{name}_patient.wav"]
    for f in files:
        subprocess.run(["git", "lfs", "pull", f"--include={f}"], cwd=ROOT, check=True)
    out = f"/content/{name}.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                    "-i", f"{ROOT}/{files[0]}", "-i", f"{ROOT}/{files[1]}",
                    "-filter_complex", "amix=inputs=2:duration=longest",
                    "-ac", "1", "-ar", "16000", "-t", str(seconds), out], check=True)
    return out

## 5. `asr_001.json` üret ve kaydet (ilk 3 dakika)

In [19]:
import json
wav = prepare_audio("day1_consultation01", seconds=180)
d = transcribe(wav, dialogue_id="asr_001", num_speakers=2)
print(check_format(d), "| cümle:", len(d["segments"]))
for s in d["segments"][:8]:
    print(f'{s["start"]:6.1f}-{s["end"]:6.1f} {s["speaker"]}: {s["text"]}')

with open("/content/asr_001.json", "w") as f:
    json.dump(d, f, ensure_ascii=False, indent=2)

True | cümle: 64
   2.9-   3.2 SPEAKER_00: Hello.
   4.0-   5.2 SPEAKER_01: Hello, how are you?
   6.0-   6.3 SPEAKER_00: Hi.
   6.9-   7.4 SPEAKER_00: Shall we stop?
   7.8-  10.7 SPEAKER_00: Yeah, OK. Hello.
  10.7-  11.9 SPEAKER_00: How can I help you this morning?
  12.7-  13.4 SPEAKER_01: Hi.
  13.6-  17.5 SPEAKER_01: I've just had some diarrhea for the last three days.


In [20]:
# Drive'a kopyala (kalıcı) ve bilgisayara indir
from google.colab import drive, files
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/MedScribe_Drive
!cp /content/asr_001.json /content/drive/MyDrive/MedScribe_Drive/
files.download("/content/asr_001.json")

Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Değerlendirme: konuşmacı ayırma doğruluğu (PriMock57, ilk 5 dakika)
Gold konuşmacı kanal enerjisinden (hangi kanal daha yüksek: doktor/hasta). SPEAKER → rol eşlemesi çoğunluk oyuyla.
Doğruluk = cümlelerin kaçında model konuşmacısı gold ile uyuşuyor. Kısa = < 1 sn.

In [21]:
import collections
import numpy as np
import pandas as pd
import soundfile as sf

def channel_gold(name, seg_list):
    """Her cümle için doktor/hasta kanalının RMS'i yüksek olan rolü döner."""
    dr, sr = sf.read(f"{ROOT}/audio/{name}_doctor.wav")
    pt, _ = sf.read(f"{ROOT}/audio/{name}_patient.wav")
    gold = []
    for s in seg_list:
        a, b = int(s["start"] * sr), int(s["end"] * sr)
        rd = np.sqrt(np.mean(dr[a:b] ** 2)) if b > a else 0
        rp = np.sqrt(np.mean(pt[a:b] ** 2)) if b > a else 0
        gold.append("Doctor" if rd >= rp else "Patient")
    return gold

def pm57_eval(name, seconds=300, num_speakers=2, whisper_model="small"):
    wav = prepare_audio(name, seconds)
    d = transcribe(wav, whisper_model=whisper_model, num_speakers=num_speakers)
    segs = d["segments"]
    gold = channel_gold(name, segs)
    votes = collections.defaultdict(collections.Counter)
    for s, g in zip(segs, gold):
        votes[s["speaker"]][g] += 1
    mapping = {spk: c.most_common(1)[0][0] for spk, c in votes.items()}
    ok = [mapping[s["speaker"]] == g for s, g in zip(segs, gold)]
    short = [o for s, o in zip(segs, ok) if s["end"] - s["start"] < 1]
    long_ = [o for s, o in zip(segs, ok) if s["end"] - s["start"] >= 1]
    mean = lambda x: round(float(np.mean(x)), 3) if x else None
    return {"muayene": name, "model": whisper_model, "konusmaci_sayisi": len(votes),
            "cumle": len(segs), "dogruluk": mean(ok),
            "kisa_n": len(short), "kisa_dogruluk": mean(short),
            "uzun_n": len(long_), "uzun_dogruluk": mean(long_),
            "dogru_n": int(sum(ok))}

In [22]:
audio_dir = f"{ROOT}/audio"
all_names = sorted({f.rsplit("_", 1)[0] for f in os.listdir(audio_dir) if f.endswith("_doctor.wav")})
names = all_names[:20]   # day1_consultation01..15 + day2_consultation01..05
print(len(all_names), "muayene var, kullanılan:", len(names))

rows = []
for n in names:
    try:
        rows.append(pm57_eval(n, seconds=300, num_speakers=2))
        pd.DataFrame(rows).to_csv("/content/eval_small.csv", index=False)
        print("tamam:", n)
    except Exception as e:
        print("ATLANDI:", n, type(e).__name__, e)
df = pd.DataFrame(rows)
df

57 muayene var, kullanılan: 20
tamam: day1_consultation01
tamam: day1_consultation02
tamam: day1_consultation03
tamam: day1_consultation04
tamam: day1_consultation05
tamam: day1_consultation06
tamam: day1_consultation07
tamam: day1_consultation08
tamam: day1_consultation09
tamam: day1_consultation10
tamam: day1_consultation11
tamam: day1_consultation12
tamam: day1_consultation13
tamam: day1_consultation14
tamam: day1_consultation15
tamam: day2_consultation01
tamam: day2_consultation02
tamam: day2_consultation03
tamam: day2_consultation04
tamam: day2_consultation05


,muayene,model,konusmaci_sayisi,cumle,dogruluk,kisa_n,kisa_dogruluk,uzun_n,uzun_dogruluk,dogru_n
0,day1_consultation01,small,2,112,0.955,39,0.923,73,0.973,107
1,day1_consultation02,small,2,97,0.918,40,0.825,57,0.982,89
2,day1_consultation03,small,2,111,0.973,35,0.971,76,0.974,108
3,day1_consultation04,small,2,89,0.933,23,0.783,66,0.985,83
4,day1_consultation05,small,2,98,1.000,28,1.000,70,1.000,98
5,day1_consultation06,small,2,99,0.970,38,0.974,61,0.967,96
6,day1_consultation07,small,2,85,0.965,18,0.889,67,0.985,82
7,day1_consultation08,small,2,121,0.942,52,0.904,69,0.971,114
8,day1_consultation09,small,2,113,0.991,47,0.979,66,1.000,112
9,day1_consultation10,small,2,99,0.949,26,0.962,73,0.945,94


In [23]:
toplam = df.cumle.sum()
print("genel doğruluk:", round(df.dogru_n.sum() / toplam, 3), "| muayene:", len(df), "| cümle:", toplam)
kisa = (df.kisa_n * df.kisa_dogruluk).sum() / df.kisa_n.sum()
uzun = (df.uzun_n * df.uzun_dogruluk).sum() / df.uzun_n.sum()
print("kısa (<1 sn):", df.kisa_n.sum(), "cümle, doğruluk", round(kisa, 3))
print("uzun (>=1 sn):", df.uzun_n.sum(), "cümle, doğruluk", round(uzun, 3))
print("3 konuşmacı bulunan muayene:", (df.konusmaci_sayisi != 2).sum())

genel doğruluk: 0.938 | muayene: 20 | cümle: 2050
kısa (<1 sn): 674 cümle, doğruluk 0.878
uzun (>=1 sn): 1376 cümle, doğruluk 0.967
3 konuşmacı bulunan muayene: 3


### 6b. small vs medium (aynı ilk 10 muayene)

In [24]:
rows_m = []
for n in names[:10]:
    try:
        rows_m.append(pm57_eval(n, seconds=300, num_speakers=2, whisper_model="medium"))
        pd.DataFrame(rows_m).to_csv("/content/eval_medium.csv", index=False)
        print("tamam:", n)
    except Exception as e:
        print("ATLANDI:", n, type(e).__name__, e)
df_m = pd.DataFrame(rows_m)

s10 = df[df.muayene.isin(df_m.muayene)]
print("small :", round(s10.dogru_n.sum() / s10.cumle.sum(), 3), "| cümle:", s10.cumle.sum())
print("medium:", round(df_m.dogru_n.sum() / df_m.cumle.sum(), 3), "| cümle:", df_m.cumle.sum())

model.bin:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

2026-11-10 18:46:57 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`


2026-11-10 18:47:00 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


tamam: day1_consultation01
tamam: day1_consultation02
tamam: day1_consultation03
tamam: day1_consultation04
tamam: day1_consultation05
tamam: day1_consultation06
tamam: day1_consultation07
tamam: day1_consultation08
tamam: day1_consultation09
tamam: day1_consultation10
small : 0.96 | cümle: 1024
medium: 0.965 | cümle: 1064


In [25]:
# tabloları Drive'a kaydet (oturum kapanırsa kaybolmasın)
!cp /content/eval_small.csv /content/eval_medium.csv /content/drive/MyDrive/MedScribe_Drive/

## 7. Kontrol deneyi: tek kişi iki rol okuyunca
Kendi sesimle (tek kişi) kaydedilen 24 sn'lik diyalog üzerinde:
- `num_speakers=None` → **tek konuşmacı** bulundu (doğru davranış: ses tek kişiye ait).
- `num_speakers=2` → bölme **keyfi** oldu; kesim cümlenin ortasında (24 sn'nin ~10. saniyesi), rollerle ilgisi yok.

Sonuç: diarization içeriğe ya da role değil, **ses tınısına** bakıyor. Bu yüzden değerlendirme için
iki gerçek konuşmacılı veri (PriMock57) kullanıldı.

## 8. Bulgular ve sınırlılıklar
Veri: PriMock57, 20 muayene, her birinin ilk 5 dakikası (temiz sahte muayeneler; kanallar elle birleştirildi).
- Genel konuşmacı doğruluğu **%93,8** (2050 cümlenin 1923'ü).
- Kısa cümleler (< 1 sn): %87,8 (674 cümle). Uzun cümleler: %96,7 (1376 cümle). Hataların yaklaşık %65'i kısa cümlelerden.
- `num_speakers=2` vermek ile vermemek gerçek iki konuşmacılı kayıtta fark yaratmadı (model 2 buldu). Tek kişilik kayıtta zorla 2 vermek keyfi bölme üretti.
- small vs medium (ilk 10 muayene): %96,0 vs %96,5. Fark gürültü düzeyinde; cümle sınırları farklı olduğu için paydalar birebir aynı değil.
- Model 3 konuşmacı bulduğunda (day2_consultation01, 04, 05) doğruluk daha düşük. Neden-sonuç test edilmedi.
- Kelime hataları ayrı bir sorun ("start" → "stop", "cough syrup"); bu tablo kelime doğruluğunu ölçmüyor.
- Gold etiket kanal enerjisinden: kanallar arası ses sızması bazı "hataları" gold'dan getirmiş olabilir.
- Çıktılar temiz kayıtta ölçüldü; telefonla alınmış gürültülü kayıtta düşmesi beklenir.
- `SPEAKER_00` / `SPEAKER_01` sıralaması rastgele; doktor/hasta kararı Kişi 2'nin işi. Tek konuşmacı bulunursa konuşmacı düzeyinde rol atama çöker, cümle düzeyinde yedek gerekli.